# Telco Customer Churn Prediction - End-to-End ML Pipeline

This notebook builds, trains, and evaluates complete Scikit-Learn pipelines using custom transformers:
- **`clean_cls`**: Automated data cleaning, type conversion, binary encoding, and dummy variables.
- **`CorrelationThresholdFilter`**: Feature selection based on Pearson correlation with the target.
- **`DecisionTreeClassifier`** & **`LogisticRegression`**: Classification models.

In [37]:
import sys
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

sys.path.append(os.path.abspath("../dataset and other libs"))
from cleaningcls import clean_cls
from ctf import CorrelationThresholdFilter
from pridict_thresh import pridict_thresh


## 1. Load Data & Train-Test Split
We pass raw features into the pipeline; the custom cleaning transformer handles all data preprocessing automatically.

In [38]:
# Load raw dataset
df = pd.read_csv('D:/Repos/Chrun-Pridictor/dataset and other libs/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Separate features (X) and target (y)
X = df.drop(columns=['Churn'])
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nTarget distribution in train set:")
print(y_train.value_counts(normalize=True).round(3))

Training set shape: (5634, 20)
Test set shape: (1409, 20)

Target distribution in train set:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


## 2. Decision Tree Pipeline
A pipeline containing `clean_cls`, `CorrelationThresholdFilter`, and `DecisionTreeClassifier`.

In [48]:
from sklearn.ensemble import RandomForestClassifier
pipeline_dt = Pipeline([
    ('clean', clean_cls()),
    ('ctf', CorrelationThresholdFilter(threshold=0.1)),
    ('pridict_thresh', pridict_thresh(RandomForestClassifier(
            n_estimators=200,           # More trees for stability
            max_depth=8,             # Prevent hyper-specific splits
            min_samples_leaf=5,         # Feature subset sampling per split
            class_weight='balanced',    # Handle churn class imbalance
            random_state=42), thresh=0.5))])    
                             # Your custom high-precision threshold
pipeline_dt.fit(X_train, y_train)
y_prob_dt = pipeline_dt.predict_proba(X_test)[:,1]
y_pred_dt = pipeline_dt.predict(X_test)
# y_pred_dt = (pipeline_dt.predict_proba(X_test)[: , 1] > 0.45).astype(int)

auc_dt = roc_auc_score(y_test, y_prob_dt)
print('=== Decision Tree Pipeline Evaluation ===')
print(f'ROC-AUC Score: {auc_dt:.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_dt))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt))

=== Decision Tree Pipeline Evaluation ===
ROC-AUC Score: 0.8423

Confusion Matrix:
[[777 258]
 [ 83 291]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.75      0.82      1035
           1       0.53      0.78      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



In [ ]:
import pandas as pd

# 1. Get the trained model from your pipeline
model = pipeline_dt['pridict_thresh'].model

# 2. Get the feature importances (scores of how much the tree uses each feature)
importances = model.feature_importances_

# 3. Get the feature names from your pipeline's preprocessing steps

feature_names = pipeline_dt['ctf'].get_feature_names_out()


# 4. Combine them into a clean DataFrame and sort by most important
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# important_f = feature_importance_df[feature_importance_df['Importance'] > 0]
active_features = feature_importance_df

print("=== Features Used by the Decision Tree (Ranked by Importance) ===")
print(active_features.to_string(index=False))

=== Features Used by the Decision Tree (Ranked by Importance) ===
                       Feature  Importance
                        tenure    0.436957
             Contract_Two year    0.247075
   InternetService_Fiber optic    0.227519
PaymentMethod_Electronic check    0.088449


In [ ]:
feature_names

Index(['tenure', 'InternetService_Fiber optic', 'Contract_Two year',
       'PaymentMethod_Electronic check'],
      dtype='object')

In [ ]:

sample_output = pipeline_dt['clean'].transform(X_train.head(5))
sample_output = pipeline_dt['ctf'].transform(sample_output)
feature_names = list(sample_output.columns)

In [ ]:
from trace_path import trace_customer_path
import trace_path

X_transformed = pipeline_dt['clean'].transform(X_train.head(1))
X_filtered = pipeline_dt['ctf'].transform(X_transformed)
trace_customer_path(pipeline_dt, X_filtered )

KeyError: None

In [ ]:
print(feature_names)
cleaned_features = pipeline_dt.named_steps['clean'].transform(X_test.reset_index(drop=True))
filtered_features = pipeline_dt.named_steps['ctf'].transform(cleaned_features.reset_index(drop=True))
for i in range(filtered_features.shape[0]):
    print(filtered_features.iloc[i].to_dict())
    print('prediction: ', y_pred_dt[i])
    print('-'*20)

['tenure', 'InternetService_Fiber optic', 'Contract_Two year', 'PaymentMethod_Electronic check']
{'tenure': 72, 'InternetService_Fiber optic': 1, 'Contract_Two year': 1, 'PaymentMethod_Electronic check': 0}
prediction:  0
--------------------
{'tenure': 8, 'InternetService_Fiber optic': 1, 'Contract_Two year': 0, 'PaymentMethod_Electronic check': 0}
prediction:  1
--------------------
{'tenure': 41, 'InternetService_Fiber optic': 0, 'Contract_Two year': 0, 'PaymentMethod_Electronic check': 0}
prediction:  0
--------------------
{'tenure': 18, 'InternetService_Fiber optic': 1, 'Contract_Two year': 0, 'PaymentMethod_Electronic check': 1}
prediction:  1
--------------------
{'tenure': 72, 'InternetService_Fiber optic': 0, 'Contract_Two year': 1, 'PaymentMethod_Electronic check': 0}
prediction:  0
--------------------
{'tenure': 21, 'InternetService_Fiber optic': 1, 'Contract_Two year': 0, 'PaymentMethod_Electronic check': 0}
prediction:  0
--------------------
{'tenure': 21, 'InternetServ

In [ ]:
y_pred_dt

array([0, 1, 0, ..., 0, 0, 0], shape=(1409,))